# Return Periods
- I will fit return periods using the method defined in the paper and then compare that to my own non-stationarity change-point analysis.


In [7]:
import xarray as xr
ds = xr.open_dataset("Caravan-nc/timeseries/netcdf/camels/camels_01013500.nc")

# Adding the water year as a coordinate
def water_year(t):
    '''
    Water year is defined as Oct 1 to Sep 30.
    Oct, Nov, Dec are labelled as regular year + 1
    '''
    return t.dt.year + (t.dt.month >= 10)

ds = ds.assign_coords(water_year=water_year(ds["date"]))
print(ds.data_vars)

Data variables:
    dewpoint_temperature_2m_max                    (date) float32 107kB ...
    dewpoint_temperature_2m_mean                   (date) float32 107kB ...
    dewpoint_temperature_2m_min                    (date) float32 107kB ...
    potential_evaporation_sum_ERA5_LAND            (date) float32 107kB ...
    potential_evaporation_sum_FAO_PENMAN_MONTEITH  (date) float32 107kB ...
    snow_depth_water_equivalent_max                (date) float32 107kB ...
    snow_depth_water_equivalent_mean               (date) float32 107kB ...
    snow_depth_water_equivalent_min                (date) float32 107kB ...
    streamflow                                     (date) float32 107kB ...
    surface_net_solar_radiation_max                (date) float32 107kB ...
    surface_net_solar_radiation_mean               (date) float32 107kB ...
    surface_net_solar_radiation_min                (date) float32 107kB ...
    surface_net_thermal_radiation_max              (date) float32 107kB 

- To begin, we need the annual peak flows.


In [17]:
annual_max = ds['streamflow'].groupby("water_year").max("date") # maximum over dates to get maximum streamflows
print(annual_max.sel(water_year=slice(1980,2008))) # 1980 to 2008 are the years of the original data


<xarray.DataArray 'streamflow' (water_year: 29)> Size: 116B
array([ 6.02,  6.62, 10.38, 14.72, 12.34,  7.88,  6.51,  9.03,  5.29,
        7.38,  9.16, 11.15,  7.93,  8.04,  8.65,  7.34, 11.69, 11.26,
       10.72,  6.57,  9.02,  8.12,  7.83,  8.37,  6.78, 14.94,  8.61,
        9.03, 19.38], dtype=float32)
Coordinates:
  * water_year  (water_year) int64 232B 1980 1981 1982 1983 ... 2006 2007 2008


- Now fit the return years to these using the LP3 distribution

In [ ]:
import numpy as np
from scipy.stats import pearson3

log_annual_max = np.log10(annual_max)
log_annual_max = log_annual_max.dropna("water_year")

params = pearson3.fit(log_annual_max)
skew, mu, sig = params # shape loc scale

0.6404974135667458 0.9515583177805591 0.12010765740133739
